# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [21]:
import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(
    api_key=os.environ.get('GEMINI_API_KEY'),
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions()),
)

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [9]:
df.info()

print('\nMissing value per kolom:')
print(df.isnull().sum())

print('\nfaq_id unik:', df['faq_id'].is_unique)
print('Pertanyaan duplikat:', df['question'].duplicated().sum())

print(f"\nJumlah kategori: {df['category'].nunique()}")
print(df['category'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   faq_id    59 non-null     int64
 1   category  59 non-null     str  
 2   question  59 non-null     str  
 3   answer    59 non-null     str  
dtypes: int64(1), str(3)
memory usage: 2.0 KB

Missing value per kolom:
faq_id      0
category    0
question    0
answer      0
dtype: int64

faq_id unik: True
Pertanyaan duplikat: 0

Jumlah kategori: 8
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [10]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [11]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [12]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


In [13]:
faiss.write_index(index, '../vector_store/faq.index')

# Verifikasi: muat kembali dari file dan cocokkan jumlah vektor dengan jumlah entri FAQ
loaded_index = faiss.read_index('../vector_store/faq.index')
print('Jumlah vektor di file index:', loaded_index.ntotal)
print('Sesuai jumlah entri knowledge base:', loaded_index.ntotal == len(df))

Jumlah vektor di file index: 59
Sesuai jumlah entri knowledge base: True


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [14]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353505)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.40880978)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


In [15]:
# Bandingkan distance: query parafrase (beda kata dengan FAQ) vs query yang tidak ada jawabannya di FAQ
for q in ['lupa kata sandi', 'bisa bayar COD?', 'bisa kirim ke luar negeri?']:
    print('Query:', q)
    for r in semantic_search(q):
        print(f"  {r['rank']}. [{r['category']}] {r['question']} (distance: {r['distance']:.3f})")
    print()

Query: lupa kata sandi
  1. [Akun] Saya lupa password akun, bagaimana cara mengaturnya ulang? (distance: 0.426)
  2. [Akun] Kenapa akun saya tiba-tiba tidak bisa login? (distance: 0.639)
  3. [Akun] Bagaimana cara mendaftar akun baru? (distance: 0.742)

Query: bisa bayar COD?
  1. [Pembayaran] Metode pembayaran apa saja yang tersedia? (distance: 0.603)
  2. [Pembayaran] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (distance: 0.609)
  3. [Promo] Bagaimana cara menggunakan kode voucher saat checkout? (distance: 0.621)

Query: bisa kirim ke luar negeri?
  1. [Pengiriman] Berapa lama estimasi waktu pengiriman barang? (distance: 0.588)
  2. [Pengiriman] Apakah bisa memilih jasa pengiriman sendiri saat checkout? (distance: 0.618)
  3. [Pemesanan] Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat? (distance: 0.618)



## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [16]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Format jawaban:
- Gunakan Bahasa Indonesia yang ramah dan mudah dipahami.
- Sampaikan inti jawaban langsung di kalimat pertama.
- Jangan menambahkan atau menyimpulkan detail yang tidak tertulis eksplisit di FAQ (misalnya angka, nama bank, nomor telepon, atau syarat lain).
- Jika informasi tidak tersedia, jawab: "Maaf, saya belum memiliki informasi tersebut. Silakan hubungi customer service kami untuk bantuan lebih lanjut."

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

In [17]:
# Lihat preview prompt utuh yang akan dikirim ke LLM 
print(build_prompt('bisa nyicil?', semantic_search('bisa nyicil?')))

Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
FAQ: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

FAQ: Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller?
Jawaban: Untuk pembelian grosir, silakan hubungi penjual langsung melalui chat untuk menanyakan ketersediaan stok dan kemungkinan harga khusus.

FAQ: Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
Jawaban: Saat ini sistem belum mendukung pembayaran gabungan (split payment) dalam satu transaksi.

Format jawaban:
- Gunakan Bahasa Indonesia yang ramah dan mudah dipahami.
- Sampaikan inti jawaban langsung di kalimat pertama.
- J

## 7. Mengintegrasikan LLM

In [19]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [22]:
test_queries = [
    # Jelas ada jawabannya di FAQ
    'Metode pembayaran apa saja yang tersedia?',
    'Bagaimana cara melacak paket yang sedang dikirim?',
    # Parafrase / sinonim / bahasa informal (kata berbeda dari FAQ asli)
    'Saldo saya sudah terpotong tapi pesanan masih belum lunas',
    'Barang yang datang pecah, harus gimana?',
    'Duit refund saya balik ke mana ya?',
    'bisa tuker size?',
    # Informasi sebagian: topiknya ada di FAQ, tapi detail yang ditanya tidak ada
    'Berapa nomor telepon customer service?',
    # Tidak ada jawabannya di FAQ
    'Bisa bayar COD?',
    'Bisa kirim ke luar negeri?',
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Metode pembayaran apa saja yang tersedia?
Jawaban: Halo! Saat ini kami mendukung metode pembayaran melalui transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan. Semoga membantu!
Sumber teratas: Metode pembayaran apa saja yang tersedia?
---
Query: Bagaimana cara melacak paket yang sedang dikirim?
Jawaban: Anda dapat melacak paket dengan menggunakan nomor resi yang muncul di halaman 'Pesanan Saya' setelah barang dikirim, lalu gunakan nomor tersebut untuk melacak posisi paket di website kurir terkait.
Sumber teratas: Bagaimana cara melacak paket yang sedang dikirim?
---
Query: Saldo saya sudah terpotong tapi pesanan masih belum lunas
Jawaban: Halo! Hal tersebut biasanya terjadi karena adanya keterlambatan sinkronisasi pada sistem pembayaran. Silakan tunggu selama 15-30 menit, dan jika status pesanan Anda belum juga berubah, silakan hubungi customer service kami untuk bantuan lebih lanjut.
Sumber teratas: Kenapa pembayaran saya berhasil dipotong tap

In [23]:
# Evaluasi retrieval per query.
# Label (tipe & FAQ yang diharapkan) ditentukan SEBELUM sistem dijalankan, urutannya sama dengan test_queries.
# None = jawabannya memang tidak ada di FAQ, sistem diharapkan memberi fallback.
labels = [
    ('Jelas', 17),
    ('Jelas', 10),
    ('Parafrase', 21),
    ('Parafrase', 16),
    ('Parafrase', 27),
    ('Parafrase', 30),
    ('Sebagian', 48),
    ('Tidak ada', None),
    ('Tidak ada', None),
]
assert len(labels) == len(test_queries)

faq_id_by_question = dict(zip(df['question'], df['faq_id']))

rows = []
for q, (tipe, expected_id) in zip(test_queries, labels):
    results = semantic_search(q)
    top_ids = [faq_id_by_question[r['question']] for r in results]
    rows.append({
        'query': q,
        'tipe': tipe,
        'harapan': f'#{expected_id}' if expected_id else '-',
        'top-3': ', '.join(f'#{i}' for i in top_ids),
        'rank harapan': top_ids.index(expected_id) + 1 if expected_id in top_ids else '-',
        'distance top-1': round(float(results[0]['distance']), 3),
        'selisih top-2 vs top-1': round(float(results[1]['distance'] - results[0]['distance']), 3),
    })

eval_df = pd.DataFrame(rows)
answerable = eval_df[eval_df['harapan'] != '-']
print('Top-1 tepat:', (answerable['rank harapan'] == 1).sum(), '/', len(answerable))
print('Masuk top-3:', (answerable['rank harapan'] != '-').sum(), '/', len(answerable))
eval_df

Top-1 tepat: 6 / 7
Masuk top-3: 7 / 7


,query,tipe,harapan,top-3,rank harapan,distance top-1,selisih top-2 vs top-1
0,Metode pembayaran apa saja yang tersedia?,Jelas,#17,"#17, #22, #1",1,0.314,0.193
1,Bagaimana cara melacak paket yang sedang dikirim?,Jelas,#10,"#10, #52, #12",1,0.260,0.171
2,Saldo saya sudah terpotong tapi pesanan masih ...,Parafrase,#21,"#21, #8, #44",1,0.319,0.159
3,"Barang yang datang pecah, harus gimana?",Parafrase,#16,"#16, #54, #24",1,0.336,0.143
4,Duit refund saya balik ke mana ya?,Parafrase,#27,"#27, #25, #24",1,0.238,0.172
5,bisa tuker size?,Parafrase,#30,"#46, #30, #58",2,0.681,0.015
6,Berapa nomor telepon customer service?,Sebagian,#48,"#48, #49, #59",1,0.272,0.191
7,Bisa bayar COD?,Tidak ada,-,"#17, #22, #37",-,0.612,0.004
8,Bisa kirim ke luar negeri?,Tidak ada,-,"#9, #15, #13",-,0.585,0.029


### Catatan Analisis Relevansi

**Ringkasan.** Ada 9 query uji: 2 jelas, 4 parafrase atau informal, 1 informasi sebagian, dan 2 yang tidak ada di FAQ.
- **Retrieval:** dari 7 query yang punya jawaban, FAQ yang diharapkan berada di **top-1 pada 6 query** dan di **top-3 pada ketujuhnya**.
- **LLM:** tidak mengarang informasi. Tidak ada nomor telepon, nama bank, atau angka yang tidak tertulis di FAQ.

#### 1. Retrieval yang akurat
- **Query jelas (#17, #10).** Distance top-1 0.26–0.31, dengan selisih ke top-2 ≥ 0.17, sehingga pemenangnya jelas. Distance tidak 0 walaupun query hampir identik dengan pertanyaan FAQ. Penyebabnya, dokumen yang di-embed berisi pertanyaan **dan** jawaban, sedangkan query hanya berisi pertanyaan.
- **Query parafrase berhasil menangkap makna, bukan kata kunci:**
  - "saldo sudah *terpotong* tapi pesanan *belum lunas*" → #21 ("pembayaran berhasil *dipotong* tapi status *belum terbayar*");
  - "barang datang *pecah*" → #16 ("paket diterima dalam kondisi *rusak*");
  - "*duit* refund *balik ke mana*" → #27.

  Distance-nya (0.24–0.34) setara dengan query jelas. Bahasa informal tidak menurunkan kualitas retrieval.
- **Query informasi sebagian** ("nomor telepon customer service") → #48 dengan distance 0.272. Retrieval-nya benar. Tantangannya ada di sisi generation (lihat poin 4).

#### 2. Retrieval yang kurang relevan
- **"bisa tuker size?"** → top-1 adalah #46 (size chart produk). FAQ yang benar, #30 (menukar ukuran/warna), berada di rank 2.
  - Distance top-1 = 0.681, **lebih besar daripada query yang tidak ada di FAQ**. Selisih top-1 dan top-2 hanya 0.015.
  - Penyebabnya: query sangat pendek (3 kata) dan informal ("tuker"), sedangkan kata "size" secara literal hanya muncul di #46. Embedding terseret oleh kemiripan kata, bukan oleh maksudnya (menukar).
- **Tertolong oleh `top_k=3`.** LLM tetap menjawab dari #30 ("Penukaran ukuran tergantung pada kebijakan masing-masing penjual..."). Ini menunjukkan kenapa mengambil lebih dari satu FAQ itu penting.
  - Catatan: loop bawaan hanya mencetak "Sumber teratas" (#46), padahal jawabannya berasal dari #30. Kalau hanya melihat top-1, kita bisa salah menyimpulkan bahwa jawabannya diambil dari FAQ yang salah.

#### 3. Pola distance: query relevan vs query di luar FAQ
- **Query di luar FAQ** (COD, luar negeri): distance top-1 0.585–0.612, dan top-3-nya nyaris datar (selisih 0.004 dan 0.029). Tidak ada FAQ yang benar-benar cocok. Yang muncul hanya FAQ yang "satu topik" (pembayaran, pengiriman).
- **Threshold distance saja tidak cukup untuk menolak query.** "bisa tuker size?" punya jawaban, tetapi distance-nya (0.681) **lebih besar** daripada COD (0.612). Threshold misalnya 0.5 justru akan menolak query yang valid ini. Karena itu, keputusan "informasi tidak tersedia" tepat diserahkan ke LLM. Distance dan selisih top-1/top-2 lebih cocok dipakai sebagai sinyal **ambiguitas**.

#### 4. Kualitas jawaban LLM
- **Query jelas dan parafrase:** jawabannya natural dan merupakan parafrase setia dari FAQ. Nuansa "biasanya" di #21 dipertahankan, dan angkanya (15–30 menit, 2x24 jam) sesuai FAQ.
- **Query di luar FAQ:** COD dan luar negeri dijawab dengan kalimat fallback, persis seperti yang diminta bagian Format. Untuk "luar negeri", LLM **tidak terjebak** menyamakan "luar pulau" (FAQ #9) dengan "luar negeri". Instruksi "jangan menyimpulkan detail yang tidak tertulis eksplisit" bekerja.
- **Trade-off pada COD:** context memuat daftar metode pembayaran (#17) yang tidak menyebut COD. LLM memilih fallback alih-alih menjawab "COD tidak tersedia". Pilihan ini **aman**, karena FAQ tidak pernah menyatakan COD tidak tersedia. Tapi jawabannya **kurang membantu**, karena LLM bisa saja menyebutkan metode yang memang tersedia.
- **Informasi sebagian** (nomor telepon CS):
  - Positif: LLM tidak mengarang nomor. LLM menjawab bahwa nomornya tertera di halaman bantuan, dan menyarankan live chat atau email. Keduanya sesuai #48.
  - Masalah: jawabannya **kontradiktif**. Kalimat pertama sudah menjawab sebagian ("tertera di halaman bantuan"), lalu disusul "Maaf, saya belum memiliki informasi spesifik...". LLM menempelkan template fallback ke jawaban yang sebenarnya sudah parsial. Kalimat fallback yang dipatok di prompt terlalu kaku untuk kasus jawaban parsial.

#### 5. Kesimpulan dan saran pengembangan
- Semantic search andal untuk query yang jelas maupun parafrase. Titik lemahnya ada pada **query yang sangat pendek atau ambigu**.
- Grounding LLM kuat: tidak ada halusinasi. Kelemahannya ada pada kekakuan fallback untuk jawaban parsial.
- **Saran:**
  1. Tambahkan instruksi khusus di prompt untuk jawaban parsial: sampaikan bagian yang ada, lalu sebutkan bagian yang tidak tersedia, tanpa template fallback.
  2. Tampilkan top-3 sumber, bukan hanya top-1.
  3. Pakai distance dan selisih top-1/top-2 untuk menandai query ambigu (misalnya dengan meminta klarifikasi), bukan untuk menolak query.
  4. Uji dengan lebih banyak query.